# RAG-Powered Review Search: "Ask Your Reviews"

## Business Question
Can a system answer specific questions about customer feedback by retrieving only the most relevant reviews and generating a grounded answer — rather than requiring someone to manually read through every review, or relying on an AI's generic guess?

## What This Demonstrates
This project implements Retrieval-Augmented Generation (RAG) from scratch, without a pre-packaged framework, to show a genuine understanding of how the technique actually works under the hood:
1. **Embedding** — converting review text into numerical vectors that capture meaning, using Google's embedding model
2. **Retrieval** — measuring cosine similarity between a question's embedding and every review's embedding to find the most relevant matches
3. **Augmented Generation** — passing only the retrieved, relevant reviews to an LLM as context, so its answer is grounded in real data rather than general knowledge

## Tools Used
Python, Google Gemini API (`gemini-embedding-001` for embeddings, `gemini-3.5-flash-lite` for generation), pandas, numpy

## Approach
1. Loaded a sample of real customer reviews (same dataset as the AI Review Analyzer project)
2. Generated an embedding for each review using Gemini's embedding model
3. Built a cosine similarity function to measure how closely two embeddings match
4. Took a natural-language question, embedded it the same way, and ranked all reviews by similarity to find the most relevant ones
5. Passed the top-ranked reviews to an LLM alongside the original question, instructing it to answer using only that retrieved context

## Example Result
**Question asked:** "What do customers complain about with sizing?"

The system correctly retrieved reviews about restricted movement, items running small, and specific sizing details (e.g. a customer going from a 6/28 to a medium) — then generated an answer citing these specific, real details rather than a generic response about sizing in general.

## Scope Note
This implementation stores embeddings in memory (a Python list) rather than a dedicated vector database (like Pinecone, Chroma, or FAISS), since the dataset here is small. The underlying retrieval logic — embedding, cosine similarity, ranking — is identical to what a production vector database does at scale; a real vector database simply makes this search fast and efficient across millions of documents instead of a few dozen.

## Files
- `rag_review_search.ipynb` — full notebook with code, retrieval logic, and example results

In [1]:
import pandas as pd

reviews = pd.read_csv('/Users/shobarathore/Downloads/Portfolio Projects/ai-review-analyze/Womens Clothing E-Commerce Reviews.csv')
sample = reviews.dropna(subset=['Review Text']).sample(15, random_state=1)
sample[['Review Text', 'Department Name']]

,Review Text,Department Name
6257,"I'm taking this to hawaii for the holidays, i'...",Dresses
7519,I purchased this sweater in the blue and i lov...,Tops
18421,Absolutely in love with this top. the embroide...,Tops
1329,"In the midst of the end-of-winter doldrums, i ...",Tops
6069,"According to the label, this dress can be hand...",Dresses
1894,I first bought the red & green at the store. w...,Tops
13505,This is a great dress that i was eyeing for so...,Dresses
16236,This is a unique blouse and well worth a try. ...,Trend
7126,Yeah i know that sounds ridiculous. but this i...,Bottoms
544,I purchased this dress because i have a simila...,Dresses


In [ ]:
from google import genai
from google.genai import types

API_KEY = input('Enter your Gemini API key: ')
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=15000))

In [3]:
test_text = sample.iloc[0]['Review Text']

result = client.models.embed_content(
    model='gemini-embedding-001',
    contents=test_text
)

embedding = result.embeddings[0].values
print(len(embedding))
print(embedding[:5])

3072
[-0.015924996, -0.026782602, 0.0010771628, -0.07946278, -0.010473721]


In [5]:
import time

embeddings = []

for i, row in sample.iterrows():
    result = client.models.embed_content(
        model='gemini-embedding-001',
        contents=row['Review Text']
    )
    embeddings.append(result.embeddings[0].values)
    print(f"Embedded {i}")
    time.sleep(2)

Embedded 6257
Embedded 7519
Embedded 18421
Embedded 1329
Embedded 6069
Embedded 1894
Embedded 13505
Embedded 16236
Embedded 7126
Embedded 544
Embedded 10800
Embedded 4078
Embedded 5246
Embedded 14043
Embedded 17101


In [6]:
import numpy as np

In [8]:
def cosine_similarity(a, b): a = np.array(a); b = np.array(b); return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [9]:
cosine_similarity(embeddings[0], embeddings[0])

np.float64(1.0000000000000002)

In [11]:
question = "What do customers complain about with sizing?"; question_result = client.models.embed_content(model='gemini-embedding-001', contents=question); question_embedding = question_result.embeddings[0].values

In [12]:
similarities = [cosine_similarity(question_embedding, emb) for emb in embeddings]; sample['similarity'] = similarities; sample.sort_values('similarity', ascending=False)[['Review Text', 'similarity']]

,Review Text,similarity
17101,I tried on this top and it looked fine until i...,0.661632
4078,Loved the look of this top. sadly it runs quit...,0.635202
6069,"According to the label, this dress can be hand...",0.609640
7126,Yeah i know that sounds ridiculous. but this i...,0.609611
16236,This is a unique blouse and well worth a try. ...,0.601959
13505,This is a great dress that i was eyeing for so...,0.593854
5246,Love pilcro jeans and really hoped these would...,0.576541
7519,I purchased this sweater in the blue and i lov...,0.573421
14043,I love this jacket. something to get cozy in o...,0.572485
18421,Absolutely in love with this top. the embroide...,0.541586


In [13]:
top_reviews = sample.sort_values('similarity', ascending=False).head(4)['Review Text'].tolist(); context = '\n\n'.join([f"Review: {r}" for r in top_reviews]); print(context)

Review: I tried on this top and it looked fine until i tried to move. you can't lift your arms. i tried on a larger size but still felt like the movement was limited in the back and arm areas.

Review: Loved the look of this top. sadly it runs quite small and was very tight where other non-swing tops in a large do not. the item ran out of stock far too quickly. i normally wear a large but feared that the large would be small and wanted to order up but it was already put of stock. the quality and style are great especially for a larger chest. length is great for taller women.

Review: According to the label, this dress can be hand washed or dry cleaned. i hand washed and line dried per the instructions on the label, and all the seams have completely shrunk and gathered. it looks like there's a drawstring running through every seam. so disappointed in the quality and the labeling. awful! i gave it one extra star because it's cute.

Review: Yeah i know that sounds ridiculous. but this is 

In [14]:
final_prompt = f"Based only on the following customer reviews, answer this question: {question}\n\n{context}"; response = client.models.generate_content(model='gemini-3.5-flash-lite', contents=final_prompt); print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the provided customer reviews, customers complain about the following sizing issues:

* **Restricted movement:** One customer stated that you cannot lift your arms and that movement is limited in the back and arm areas, even in a larger size.
* **Running small:** Multiple customers noted that the items run small and are very tight compared to normal sizes (one noted it runs quite small, and another mentioned that a medium felt very form-fitting for someone usually a size 6/28).
